# This copy of the script uses q2 for line-by-line differential atmospheric parameter determination.
### It replaces ispec.model_spectrum_from_ew() in the parameter iteration.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import logging
import multiprocessing
from multiprocessing import Pool
import matplotlib.pyplot as plt
from scipy.stats import norm
from userlist import *
import shutil
import q2

# Part I: Check Gaussian fits of Fe lines before atmospheric parameter iteration
### example.py: find_linemasks()

In [ ]:
# =============== 1. Get paths & read data =================
# Get the spectrum and linelist paths
star_spectrum_file = spectrum_norm_path
star_spectrum = ispec.read_spectrum(star_spectrum_file)

# Copy the linelist file to a new working location
shutil.copyfile(output_linelist_path, linelist_target_path)
print(f"Copied {output_linelist_path} -> {linelist_target_path}")
atomic_linelist_file = linelist_target_path
print(atomic_linelist_file)

# =============== 2. Read linelist and restrict wavelength range =================
# Read linelist (start line-mask finding)
logging.info("Finding line masks...")
atomic_linelist = ispec.read_atomic_linelist(atomic_linelist_file)  # Read full linelist (optional)
print(f"Successfully read {len(atomic_linelist)} lines from the linelist.")

# Filter linelist to the actual wavelength coverage of the observed spectrum.
# For example, if star_spectrum spans 480–680 nm, keep only lines within this range.
atomic_linelist = ispec.read_atomic_linelist(
    atomic_linelist_file,
    wave_base=np.min(star_spectrum['waveobs']),
    wave_top=np.max(star_spectrum['waveobs'])
)

# atomic_linelist = atomic_linelist[atomic_linelist['theoretical_depth'] >= 0.01]
# Select lines that have a minimum contribution in the solar spectrum (optional)

print(
    f"Successfully read {len(atomic_linelist)} lines from the linelist in the range "
    f"{np.min(star_spectrum['waveobs'])}–{np.max(star_spectrum['waveobs'])}."
)
print(
    "Linelist wavelength range:",
    np.min(atomic_linelist['wave_nm']),
    "to",
    np.max(atomic_linelist['wave_nm']),
    "nm."
)

In [ ]:
# =============== Spectral convolution (smoothing) + continuum modeling ===============
if from_resolution <= to_resolution:
    print("from_resolution is less than to_resolution, no need to convolve.")
    smoothed_star_spectrum = star_spectrum
else:
    # convolve_spectrum is used to simulate the instrumental spectral resolution
    smoothed_star_spectrum = ispec.convolve_spectrum(star_spectrum, to_resolution)

# The spectrum is already normalized, so no actual fitting is needed;
# we simply provide a fixed continuum model at unity (1.0).
star_continuum_model = ispec.fit_continuum(
    star_spectrum,
    fixed_value=1.0,
    model="Fixed value"
)

# Telluric settings
telluric_linelist = None
vel_telluric = 0.0
min_depth = 0.05    # Minimum absorption depth for line finding (not critical here)
max_depth = 1.00

# =============== Line finding + Gaussian fitting ===============
# atomic_linelist: atomic line list used for cross-matching
# max_atomic_wave_diff: allowed line-center matching tolerance (in nm)
# discard_voigt=True: use Gaussian profiles only
# min_depth=0.05: consider only lines deeper than 5%
# closest_match=False: match lines using theoretical parameters
star_linemasks = ispec.find_linemasks(
    star_spectrum,
    star_continuum_model,
    atomic_linelist=atomic_linelist,
    max_atomic_wave_diff=0.005,
    telluric_linelist=telluric_linelist,
    vel_telluric=vel_telluric,
    minimum_depth=min_depth,
    maximum_depth=max_depth,
    smoothed_spectrum=smoothed_star_spectrum,
    check_derivatives=False,
    discard_gaussian=False,
    discard_voigt=True,
    closest_match=False
)

# star_linemasks is a structured array containing the fitted line parameters
# (mu, sigma, amplitude, etc.), equivalent width (ew), rms, depth, as well as
# cross-matched information such as element name, ionization stage, loggf, etc.

In [ ]:
# =============== Filter invalid spectral lines ===============
# Exclude lines that were not successfully cross-matched with the atomic linelist
# (wave_nm == 0 indicates no atomic match, and such lines cannot be used for abundance analysis)
rejected_by_atomic_line_not_found = (star_linemasks['wave_nm'] == 0)
star_linemasks = star_linemasks[~rejected_by_atomic_line_not_found]

# Exclude lines with zero equivalent width
# (EW = 0 usually indicates a failed fit or no detectable absorption feature)
rejected_by_zero_ew = (star_linemasks['ew'] == 0)
star_linemasks = star_linemasks[~rejected_by_zero_ew]

# ========== Keep only lines with EW between 20 and 100 mÅ ==========
ew_range_mask = (star_linemasks['ew'] >= 20) & (star_linemasks['ew'] <= 100)
star_linemasks = star_linemasks[ew_range_mask]

# =============== Optional: select Fe lines only (for iron abundance analysis) ===============
iron = (star_linemasks['element'] == "Fe 1")
iron = np.logical_or(iron, star_linemasks['element'] == "Fe 2")
iron_star_linemasks = star_linemasks[iron]

# =============== Write linemask files ===============
linemask_output_folder = output_folder + "/linemasks"
os.makedirs(linemask_output_folder, exist_ok=True)

# Write regions with mask limits only
ispec.write_line_regions(
    star_linemasks,
    linemask_output_folder + "/" + target + "_melendez2014_star_linemasks.txt"
)

# Write iron-only regions with mask limits
ispec.write_line_regions(
    iron_star_linemasks,
    linemask_output_folder + "/" + target + "_melendez2014_star_fe_linemasks.txt"
)

# Write regions with mask limits, cross-matched atomic data, and fitted parameters
ispec.write_line_regions(
    star_linemasks,
    linemask_output_folder + "/" + target + "_melendez2014_star_fitted_linemasks.txt",
    extended=True
)
recover_star_linemasks = ispec.read_line_regions(
    linemask_output_folder + "/" + target + "_melendez2014_star_fitted_linemasks.txt"
)

# Write regions with mask limits and atomic data, but with fitted parameters reset to zero
zeroed_star_linemasks = ispec.reset_fitted_data_fields(star_linemasks)
ispec.write_line_regions(
    zeroed_star_linemasks,
    linemask_output_folder + "/" + target + "_melendez2014_star_zeroed_fitted_linemasks.txt",
    extended=True
)

# Write atomic linelist corresponding to the selected regions only
ispec.write_atomic_linelist(
    star_linemasks,
    linemask_output_folder + "/" + target + "_melendez2014_star_atomic_linelist.txt"
)

- make the plot

In [ ]:
# =============== Plot Gaussian fits for Fe lines ===============
# Create output directories
output_dir = output_folder + "/figs_Fe_GaussianFits"
os.makedirs(output_dir, exist_ok=True)
deleted_felines_folder_path = output_dir + "/deleted"
os.makedirs(deleted_felines_folder_path, exist_ok=True)

# Keep only Fe I and Fe II lines
fe_lines = star_linemasks[
    (star_linemasks['element'] == "Fe 1") |
    (star_linemasks['element'] == "Fe 2")
]

w_range = 0.25  # nm

# Define Gaussian model
def gaussian(x, mu, sig, A, baseline):
    return baseline + A * np.exp(-(x - mu)**2 / (2 * sig**2))

# Loop over each Fe line and generate plots
for idx, line in enumerate(fe_lines):
    mu = line['mu']
    sig = line['sig']
    A = line['A']
    baseline = line['baseline']
    
    # Skip invalid fits
    if sig == 0 or mu == 0:
        continue

    # Extract wavelength window from the original spectrum
    mask = (
        (star_spectrum['waveobs'] >= mu - w_range) &
        (star_spectrum['waveobs'] <= mu + w_range)
    )
    wave = star_spectrum['waveobs'][mask]
    flux = star_spectrum['flux'][mask]

    # Build fitted curve
    fit_x = np.linspace(mu - w_range, mu + w_range, 300)
    fit_y = gaussian(fit_x, mu, sig, A, baseline)

    # Plot
    plt.figure(figsize=(8, 5), dpi=128)
    plt.plot(wave, flux, label='Observed Spectrum', color='blue', lw=0.7)
    plt.plot(fit_x, fit_y, '--', label='Gaussian Fit', color='red')
    plt.axvline(mu, color='orange', linestyle=':', label=f"$\\mu$ = {mu:.3f} nm")
    plt.title(f"Gaussian Fit: {line['element']} {line['wave_A']:.2f} Å", fontsize=16)
    plt.xlabel("Wavelength (nm)", fontsize=16)
    plt.ylabel("Normalized Flux", fontsize=16)
    plt.xticks(fontsize=13)
    plt.yticks(fontsize=13)
    plt.ticklabel_format(style='plain', axis='x')

    # Annotation box
    text = (
        f"log(gf) = {line['loggf']:.2f}\n"
        f"EP = {line['lower_state_eV']:.2f} eV\n"
        f"EW = {line['ew']:.1f} mÅ"
    )
    plt.text(
        0.75, 0.05, text,
        transform=plt.gca().transAxes,
        fontsize=13,
        bbox=dict(facecolor='white', alpha=0.8)
    )
    plt.legend(loc="lower left", fontsize=13)
    plt.tight_layout()
    
    # Save figure
    out_path = os.path.join(
        output_dir,
        f"FeFit_{idx+1}_{line['element']}_{line['wave_A']:.2f}.png"
    )
    plt.savefig(out_path)
    plt.close()

print(f"Plotting completed: {len(fe_lines)} figures saved in {output_dir}/")

### Create tables for manually recording deleted or modified spectral lines

In [ ]:
# Copy the linelist used for this target into the linemasks folder for reference
shutil.copyfile(
    output_linelist_path,
    output_folder + "linemasks/linelist_for_" + target + "_copied.tsv"
)

deleted_file_path  = output_folder + "linemasks/Lines_deleted.tsv"
modified_file_path = output_folder + "linemasks/Lines_EW_modified.tsv"

# Create linemasks directory if it does not exist
os.makedirs(os.path.dirname(deleted_file_path), exist_ok=True)
os.makedirs(os.path.dirname(modified_file_path), exist_ok=True)

# Full header for Lines_deleted.tsv
cols_deleted = [
    "element","wave_A","wave_nm","loggf","lower_state_eV","lower_state_cm1","lower_j",
    "upper_state_eV","upper_state_cm1","upper_j","upper_g","lande_lower","lande_upper",
    "spectrum_transition_type","turbospectrum_rad","rad","stark","waals",
    "waals_single_gamma_format","turbospectrum_fdamp","spectrum_fudge_factor",
    "theoretical_depth","theoretical_ew","lower_orbital_type","upper_orbital_type",
    "molecule","spectrum_synthe_isotope","ion","spectrum_moog_species",
    "turbospectrum_species","width_species","reference_code","spectrum_support",
    "turbospectrum_support","moog_support","width_support","synthe_support","sme_support"
]

# Header for Lines_EW_modified.tsv
cols_modified = ["element", "wave_A", "wave_nm", "loggf", "ew_mA_new"]

def ensure_tsv_with_header(path, columns):
    """Create an empty TSV file with the given header if it does not exist."""
    if not os.path.exists(path):
        pd.DataFrame(columns=columns).to_csv(path, sep="\t", index=False)
        print(f"Created empty file with header: {path}")
    else:
        print(f"File already exists: {path}")

ensure_tsv_with_header(deleted_file_path,  cols_deleted)
ensure_tsv_with_header(modified_file_path, cols_modified)

print("\nIf you manually deleted or modified any lines in the linemasks files,")
print(f"please record the changes in:\n  {deleted_file_path}\n  {modified_file_path}")
print("If no manual changes were made, you can safely ignore this message.")

### Manually remove poorly fitted Fe lines (and lines of other elements)
- Notes: When editing `Lines_deleted.tsv` for the second time (typically for non-Fe elements), you may notice extra tab characters in `linelist_for_target.tsv`. In that case, it is safer to copy/paste from `linemasks/linelist_for_target_copied.tsv` instead.

In [ ]:
# Original linelist file
atomic_linelist_file = linelist_target_path
# Output linelist file (overwrite the original)
filtered_linelist_file = linelist_target_path

# 1. Read the original linelist
atomic_linelist = ispec.read_atomic_linelist(atomic_linelist_file)
print("Original linelist length:", len(atomic_linelist))

# 2. Define wavelengths (nm) of lines to be removed
# Read the table of lines marked for deletion
df_deleted = pd.read_csv(output_folder + "linemasks/Lines_deleted.tsv", sep="\t")

# Normalize columns and drop invalid values
df_deleted["element"] = df_deleted["element"].astype(str)
df_deleted["wave_nm"] = pd.to_numeric(df_deleted["wave_nm"], errors="coerce")
df_deleted = df_deleted.dropna(subset=["wave_nm"])

# Separate Fe and non-Fe lines
is_fe = df_deleted["element"].str.startswith("Fe")
fe_lines    = df_deleted.loc[is_fe,  "wave_nm"].round(4).tolist()
other_lines = df_deleted.loc[~is_fe, "wave_nm"].round(4).tolist()

# Build the final removal list (Fe first, then others; unique and order-preserving)
from collections import OrderedDict
lines_to_remove = list(OrderedDict.fromkeys(fe_lines + other_lines))

# 3. Apply a boolean mask to keep lines not listed for removal
mask = ~np.isin(
    np.round(atomic_linelist["wave_nm"], 4),
    np.round(lines_to_remove, 4)
)
filtered_linelist = atomic_linelist[mask]
print("Filtered linelist length:", len(filtered_linelist))

# 4. Write the filtered linelist back to file
ispec.write_atomic_linelist(filtered_linelist, filtered_linelist_file)
print(f"New linelist saved to: {filtered_linelist_file}")

# 5. Verify that the new linelist can be read correctly
test_linelist = ispec.read_atomic_linelist(filtered_linelist_file)
print("New linelist read successfully, number of lines:", len(test_linelist))

atomic_linelist_file = linelist_target_path

### write the linelist again

In [ ]:
# =============== 2. Read the linelist and restrict the wavelength range =================
# Read the atomic linelist (initial load)
logging.info("Finding line masks...")
atomic_linelist = ispec.read_atomic_linelist(atomic_linelist_file)
print(f"Successfully read {len(atomic_linelist)} lines from the linelist.")

# Restrict the linelist to the wavelength coverage of the observed spectrum
atomic_linelist = ispec.read_atomic_linelist(
    atomic_linelist_file,
    wave_base=np.min(star_spectrum["waveobs"]),
    wave_top=np.max(star_spectrum["waveobs"])
)
print(
    f"Successfully read {len(atomic_linelist)} lines from the linelist "
    f"in the range {np.min(star_spectrum['waveobs'])}–{np.max(star_spectrum['waveobs'])}."
)
print(
    "Linelist wavelength range:",
    np.min(atomic_linelist["wave_nm"]),
    "to",
    np.max(atomic_linelist["wave_nm"]),
    "nm."
)

# =============== Line detection and Gaussian fitting =================
# atomic_linelist: atomic line list used for cross-matching
# max_atomic_wave_diff: allowed mismatch in line center (nm)
# discard_voigt=True: use Gaussian profiles only
# minimum_depth=0.05: consider lines deeper than 5%
# closest_match=False: match using theoretical parameters
star_linemasks = ispec.find_linemasks(
    star_spectrum,
    star_continuum_model,
    atomic_linelist=atomic_linelist,
    max_atomic_wave_diff=0.005,
    telluric_linelist=telluric_linelist,
    vel_telluric=vel_telluric,
    minimum_depth=min_depth,
    maximum_depth=max_depth,
    smoothed_spectrum=smoothed_star_spectrum,
    check_derivatives=False,
    discard_gaussian=False,
    discard_voigt=True,
    closest_match=False,
)

# star_linemasks is a structured array containing fitted line parameters
# (mu, sigma, amplitude, baseline), equivalent width (ew), rms, depth,
# element/ion identification, loggf, and other cross-matched atomic data.

# =============== Filter invalid lines =================
# Remove lines that failed atomic cross-matching (wave_nm == 0)
rejected_by_atomic_line_not_found = star_linemasks["wave_nm"] == 0
star_linemasks = star_linemasks[~rejected_by_atomic_line_not_found]

# Remove lines with zero equivalent width
rejected_by_zero_ew = star_linemasks["ew"] == 0
star_linemasks = star_linemasks[~rejected_by_zero_ew]

# Keep only lines with EW between 20 and 100 mÅ
ew_range_mask = (star_linemasks["ew"] >= 20) & (star_linemasks["ew"] <= 100)
star_linemasks = star_linemasks[ew_range_mask]

# =============== Optional: select Fe lines only =================
# Useful for iron-only abundance analysis
iron = (star_linemasks["element"] == "Fe 1") | (star_linemasks["element"] == "Fe 2")
iron_star_linemasks = star_linemasks[iron]

# =============== Write linemask files =================
linemask_output_folder = output_folder + "/linemasks"
os.makedirs(linemask_output_folder, exist_ok=True)

# Write basic line regions (mask limits only)
ispec.write_line_regions(
    star_linemasks,
    linemask_output_folder + "/" + target + "_melendez2014_star_linemasks.txt",
)

# Write Fe-only line regions
ispec.write_line_regions(
    iron_star_linemasks,
    linemask_output_folder + "/" + target + "_melendez2014_star_fe_linemasks.txt",
)

# Write line regions with cross-matched atomic data and fit results
ispec.write_line_regions(
    star_linemasks,
    linemask_output_folder + "/" + target + "_melendez2014_star_fitted_linemasks.txt",
    extended=True,
)
recover_star_linemasks = ispec.read_line_regions(
    linemask_output_folder + "/" + target + "_melendez2014_star_fitted_linemasks.txt"
)

# Write line regions with cross-matched atomic data but zeroed fit fields
zeroed_star_linemasks = ispec.reset_fitted_data_fields(star_linemasks)
ispec.write_line_regions(
    zeroed_star_linemasks,
    linemask_output_folder + "/" + target + "_melendez2014_star_zeroed_fitted_linemasks.txt",
    extended=True,
)

# Write the atomic linelist corresponding to the selected regions
ispec.write_atomic_linelist(
    star_linemasks,
    linemask_output_folder + "/" + target + "_melendez2014_star_atomic_linelist.txt",
)

# Part II: Calculate the abundances of Fe I and Fe II
### example.py: determine_abundances_from_ew()
- code = "spectrum" --> "moog"

In [ ]:
# ========== Read spectrum ==========
# --- Spectrum already generated previously ---
# star_spectrum: normalized spectrum
# star_continuum_model: continuum model obtained earlier using fit_continuum

# ========== Read saved fitted linemasks ==========
linemasks = ispec.read_line_regions(
    output_folder + f"/linemasks/{target}_melendez2014_star_fitted_linemasks.txt"
)

# Filter invalid lines
linemasks = linemasks[linemasks['wave_nm'] > 0]   # successfully cross-matched lines
linemasks = linemasks[linemasks['ew'] > 0]        # lines with valid EW

# --- Determining abundances by EW of the previously fitted lines ---
code = "moog"

# Parameters
teff = initial_teff
logg = initial_logg
MH = initial_MH
alpha = 0.00
microturbulence_vel = 1.0

# ========== Load atmosphere models and solar abundances ==========
# Selected atmosphere model grid and solar abundance set
# Different model grids correspond to different authors; differences are mainly in interpolation speed
#model = ispec_dir + "/input/atmospheres/MARCS/"          # large grid
model = ispec_dir + "/input/atmospheres/MARCS.GES/"      # smaller grid with pre-interpolation
#model = ispec_dir + "/input/atmospheres/MARCS.APOGEE/"
#model = ispec_dir + "/input/atmospheres/ATLAS9.APOGEE/"
#model = ispec_dir + "/input/atmospheres/ATLAS9.Castelli/"
#model = ispec_dir + "/input/atmospheres/ATLAS9.Kurucz/"
#model = ispec_dir + "/input/atmospheres/ATLAS9.Kirby/"

# Solar abundance reference (depends on atmosphere grid)
if "ATLAS" in model:
    solar_abundances_file = ispec_dir + "/input/abundances/Grevesse.1998/stdatom.dat"
else:
    # MARCS
    solar_abundances_file = ispec_dir + "/input/abundances/Grevesse.2007/stdatom.dat"
#solar_abundances_file = ispec_dir + "/input/abundances/Asplund.2005/stdatom.dat"
#solar_abundances_file = ispec_dir + "/input/abundances/Asplund.2009/stdatom.dat"
#solar_abundances_file = ispec_dir + "/input/abundances/Anders.1989/stdatom.dat"

# ========== Load models ==========
# Load atmosphere model grid
modeled_layers_pack = ispec.load_modeled_layers_pack(model)
# Load solar abundances
solar_abundances = ispec.read_solar_abundances(solar_abundances_file)

# Validate parameters
# Check whether parameters fall within the model grid
if not ispec.valid_atmosphere_target(
    modeled_layers_pack,
    {'teff': teff, 'logg': logg, 'MH': MH, 'alpha': alpha}
):
    msg = (
        "The specified effective temperature, gravity (log g), "
        "and metallicity [M/H] fall outside the atmospheric model grid."
    )
    print(msg)

# Prepare atmosphere model
# Interpolate atmosphere layers for the given parameters
atmosphere_layers = ispec.interpolate_atmosphere_layers(
    modeled_layers_pack,
    {'teff': teff, 'logg': logg, 'MH': MH, 'alpha': alpha},
    code=code
)

# ========== Run abundance determination ==========
spec_abund, normal_abund, x_over_h, x_over_fe = ispec.determine_abundances(
    atmosphere_layers,
    teff, logg, MH, alpha,
    linemasks,
    solar_abundances,
    microturbulence_vel=microturbulence_vel,
    verbose=1,
    code=code
)

# ========== Summary of Fe abundances ==========
bad = np.isnan(x_over_h)
fe1 = linemasks['element'] == "Fe 1"
fe2 = linemasks['element'] == "Fe 2"

fe1_abund = x_over_h[np.logical_and(fe1, ~bad)]
fe2_abund = x_over_h[np.logical_and(fe2, ~bad)]

print("\n===== [Fe/H] reference =====")
print(f"[Fe/H]: {MH} +- {row_target['[Fe/H]_err']}")

print("\n===== Fe I =====")
print("Number of Fe I lines:", len(fe1_abund))
print(
    "[Fe 1/H] median: %.4f, mean: %.4f, std: %.4f"
    % (np.median(fe1_abund), np.mean(fe1_abund), np.std(fe1_abund))
)
print(fe1_abund)

print("\n===== Fe II =====")
print("Number of Fe II lines:", len(fe2_abund))
print(
    "[Fe 2/H] median: %.4f, mean: %.4f, std: %.4f"
    % (np.median(fe2_abund), np.mean(fe2_abund), np.std(fe2_abund))
)
print(fe2_abund)

spec_abund_fe1 = spec_abund[np.logical_or(fe1, fe2)]
print(len(spec_abund_fe1), "Fe lines used for abundance analysis.")

In [ ]:
# Find indices where [Fe/H] equals a target value (within tolerance)
target_abund = 0.24

fe2_mask = (linemasks['element'] == "Fe 1") & (~np.isnan(x_over_h))
fe2_indices = np.where(fe2_mask)[0]
target_idx = fe2_indices[np.isclose(x_over_h[fe2_mask], target_abund, atol=1e-4)]

# Print the corresponding line information
if len(target_idx) > 0:
    print(linemasks[target_idx][['wave_nm', 'element', 'ew', 'loggf']])
else:
    print("No matching line found")

In [ ]:
# ========== General plotting functions ==========
def plot_abundance_trend(x, y, xlabel, ylabel, title, label, outfile=None):
    from scipy.stats import linregress
    plt.figure(figsize=(6, 4), dpi=120)
    plt.scatter(x, y, c='blue', label=label, s=40)
    slope, intercept, r_value, p_value, std_err = linregress(x, y)
    x_fit = np.linspace(min(x), max(x), 100)
    plt.plot(x_fit, slope * x_fit + intercept, 'r--', label=f"Slope = {slope:.3f}")
    plt.xlabel(xlabel, fontsize=13)
    plt.ylabel(ylabel, fontsize=13)
    plt.title(title, fontsize=14)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    if outfile:
        plt.savefig(outfile)
    else:
        plt.show()

# ========== 创建 mask ==========
fe1_mask = (linemasks['element'] == "Fe 1") & (~np.isnan(x_over_h))
fe2_mask = (linemasks['element'] == "Fe 2") & (~np.isnan(x_over_h))
fe1_abund = x_over_h[fe1_mask]
fe2_abund = x_over_h[fe2_mask]
#outfile_Fe_EW_folder = output_folder + "figs_atmos_params_Fe_EW"
#os.makedirs(outfile_Fe_EW_folder, exist_ok=True)

# ========== Fe I ==========
plot_abundance_trend(
    linemasks['lower_state_eV'][fe1_mask],
    fe1_abund,
    xlabel='Excitation Potential (eV)',
    ylabel='[Fe I/H]',
    title='[Fe I/H] vs. Excitation Potential',
    label='Fe I',
    #outfile=outfile_Fe_EW_folder+"/Fe 1_abundance_vs_EP_fit.png"
)

rew_fe1 = np.log10(linemasks['ew'][fe1_mask] / (linemasks['wave_nm'][fe1_mask] * 10))  # wave_nm to Å
plot_abundance_trend(
    rew_fe1,
    fe1_abund,
    xlabel='log(EW/λ)',
    ylabel='[Fe I/H]',
    title='[Fe I/H] vs. log(REW)',
    label='Fe I'
)

# ========== Fe II ==========
plot_abundance_trend(
    linemasks['lower_state_eV'][fe2_mask],
    fe2_abund,
    xlabel='Excitation Potential (eV)',
    ylabel='[Fe II/H]',
    title='[Fe II/H] vs. Excitation Potential',
    label='Fe II'
)

rew_fe2 = np.log10(linemasks['ew'][fe2_mask] / (linemasks['wave_nm'][fe2_mask] * 10))  # wave_nm to Å
plot_abundance_trend(
    rew_fe2,
    fe2_abund,
    xlabel='log(EW/λ)',
    ylabel='[Fe II/H]',
    title='[Fe II/H] vs. log(REW)',
    label='Fe II'
)

# Part III: Atmospheric parameters fitting

In [ ]:
#--- Read the Normalized spectrum -------------------------------------------------------------
spectrum_path = spectrum_norm_path
normalized_star_spectrum = ispec.read_spectrum(spectrum_path)

from AbundProcessFunctions import *
initial_vmic = ispec.estimate_vmic(initial_teff, initial_logg, initial_MH)

In [ ]:
# 0) Keep only Fe lines (note that iSpec uses 'Fe 1' / 'Fe 2')
elem_col = np.array([str(e).strip() for e in linemasks["element"]])
linemasks_fe = linemasks[(elem_col == "Fe 1") | (elem_col == "Fe 2")]

# 1) Specify the Sun dump file corresponding to the instrument
#    In 0_Sun_abundance.ipynb, this is saved as:
#    ispec_dir + "output/Sun/<solar_label>/atmos_params_<solar_label>.dump"
sun_dump_file = os.path.join(
    ispec_dir, "output", "Sun", solar_label, f"atmos_params_{solar_label}.dump"
)

# 2) Prepare the working directory for q2 (recommended: one per target)
q2_dir = os.path.join(output_folder, "q2_work")
os.makedirs(q2_dir, exist_ok=True)

# 3) Export q2 inputs (returns sun_par as well)
lines_csv, stars_csv, sun_par = export_q2_inputs_wide(
    ispec=ispec,
    target_id=str(target),
    used_linemasks_fe=linemasks_fe,
    solar_lines_csv=solar_lines_csv,
    instrument=instrument_name,
    resolution=from_resolution,
    out_dir=q2_dir,
    sun_dump_file=sun_dump_file,
    sun_id="Sun",
    wave_tol=0.001
)

# 4) Write stars_q2.csv (Sun uses dump values; target uses initial guesses)
stars_df = pd.DataFrame([
    {"id": "Sun",
     "teff": sun_par["teff"],
     "logg": sun_par["logg"],
     "feh":  sun_par["feh"],
     "vt":   sun_par["vt"]},
    {"id": str(target),
     "teff": float(initial_teff),
     "logg": float(initial_logg),
     "feh":  float(initial_MH),
     "vt":   float(initial_vmic)}
])
stars_df.to_csv(stars_csv, index=False)

# 5) Run q2
#    q2.Data is more robust with relative paths, so change into q2_dir
os.chdir(q2_dir)
data = q2.Data("stars_q2.csv", "lines_q2.csv")
print(data)

sp = q2.specpars.SolvePars(grid="marcs")   # use 'marcs' for MARCS atmospheres
solution_csv = os.path.join(q2_dir, f"solution_q2_{target}.csv")

q2.specpars.solve_all(data, sp, solution_csv, reference_star="Sun")

# 6) Read back the solution
sol = pd.read_csv(solution_csv)
row = sol.loc[sol["id"] == str(target)].iloc[0]

teff_q2 = float(row["teff"])
logg_q2 = float(row["logg"])
feh_q2  = float(row["feh"])   # q2 may also provide feh_model
vt_q2   = float(row["vt"])

print("[q2 solution]", target, teff_q2, logg_q2, feh_q2, vt_q2)
print("ref params:", f"Teff = {initial_teff}, logg = {initial_logg}, MH = {initial_MH}")

# Saving

In [ ]:
##--- Save results -------------------------------------------------------------
logging.info("Saving results...")

# --- 6.1 Uncertainty fields from the q2 solution ---
err_teff_q2 = float(row["err_teff"]) if "err_teff" in row and pd.notna(row["err_teff"]) else np.nan
err_logg_q2 = float(row["err_logg"]) if "err_logg" in row and pd.notna(row["err_logg"]) else np.nan

# In q2, the [Fe/H] uncertainty may appear under two possible column names:
# err_feh or err_feh_ (both exist in some outputs)
err_feh_candidates = []
for k in ["err_feh", "err_feh_"]:
    if k in row and pd.notna(row[k]):
        err_feh_candidates.append(float(row[k]))
err_feh_q2 = err_feh_candidates[0] if len(err_feh_candidates) > 0 else np.nan

err_vt_q2 = float(row["err_vt"]) if "err_vt" in row and pd.notna(row["err_vt"]) else np.nan

# --- 6.2 Assemble iSpec-style params / errors / status ---
# iSpec uses MH / vmic, while q2 uses feh / vt
alpha_keep = 0.0

params_q2 = {
    "teff": int(round(teff_q2)),
    "logg": float(logg_q2),
    "MH":   float(feh_q2),
    "alpha": alpha_keep,
    "vmic": float(vt_q2),
}

errors_q2 = {
    "teff": float(err_teff_q2),
    "logg": float(err_logg_q2),
    "MH":   float(err_feh_q2),
    "alpha": np.nan,          # q2 does not solve for alpha; leave empty
    "vmic": float(err_vt_q2),
}

status_q2 = {
    "method": "q2_line_by_line_differential",
    "reference_star": "Sun",
    "grid": "marcs",
    "q2_solution_csv": os.path.basename(solution_csv),
}

# --- 6.3 Other return slots (for compatibility with restore_results in 4_ele_abundance.ipynb) ---
# In 4_ele_abundance.ipynb, you unpack as:
# params, errors, status_fe, x_over_h_fe, selected_x_over_h_fe, fitted_lines_param_fe, used_linemasks_fe, spec_abund_fe = restore_results(...)
# Therefore, the dump must store an 8-tuple. Missing items are set to None or reused where appropriate.
x_over_h_fe = None
selected_x_over_h_fe = None
fitted_lines_param_fe = None
used_linemasks_fe = linemasks_fe          # Fe linemasks used here (preferably the filtered set)
spec_abund_fe = None

dump_payload = (
    params_q2,
    errors_q2,
    status_q2,
    x_over_h_fe,
    selected_x_over_h_fe,
    fitted_lines_param_fe,
    used_linemasks_fe,
    spec_abund_fe
)

# --- 6.4 Write dump file ---
# Recommended: do not overwrite the original iSpec dump; save as *_q2.dump
dump_file_q2 = output_atmos_params_dumpfile_path.replace(".dump", f"_q2.dump")
ispec.mkdir_p(os.path.dirname(dump_file_q2))
ispec.save_results(dump_file_q2, dump_payload)

print("Saved q2 dump ->", dump_file_q2)
print("params_q2:", params_q2)
print("errors_q2:", errors_q2)